In [1]:
# Standard library
import os
import glob
import pickle
import numpy as np
import pandas as pd

# Geospatial
import rasterio as rio
from rasterio.features import rasterize,geometry_mask
import geopandas as gpd


# Spatial processing
from scipy.ndimage import distance_transform_edt

# Machine learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)
from sklearn.preprocessing import StandardScaler

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    precision_recall_curve,
    confusion_matrix
)

In [2]:
# File Paths

# GAO reference species classifications
guava_path = "guava_20m.tif"
ohia_path = "ohia_20m.tif"

# Coastline shapefile
shapefile_path = "coast.shp"

# Sentinel-2 imagery
sentinel_path = ["sentinel/sentinel_19.tif",
                 "sentinel/sentinel_25.tif"]


# Load Species Distribution Data

In [4]:
with rio.open(guava_path) as guava_src, \
     rio.open(ohia_path) as ohia_src:

    guava = guava_src.read(1).astype(float)
    ohia = ohia_src.read(1).astype(float)

    ref_crs = guava_src.crs
    ref_transform = guava_src.transform
    
# Mask Data
guava_valid = (~np.isnan(guava)) & (guava > 0.1)
ohia_valid = (~np.isnan(ohia)) & (ohia > 0.1)
combined_mask = guava_valid | ohia_valid

# coastline buffer mask
coast_gdf = gpd.read_file(shapefile_path)

if coast_gdf.crs != ref_crs:
    coast_gdf = coast_gdf.to_crs(ref_crs)

## rasterize coast
coast_mask = rasterize(
    [(geom, 1) for geom in coast_gdf.geometry],
    out_shape=guava.shape,
    transform=ref_transform,
    fill=0,
    all_touched=True,
    dtype="uint8"
)
    

# Distance to coast in pixels
dist_pixels = distance_transform_edt(1 - coast_mask)

buffer_dist = 140  # 140 pixels = 2,800 m at 20 m resolution
buffer_mask = dist_pixels > buffer_dist


# apply final mask
final_mask = combined_mask & buffer_mask

guava_masked = np.where(final_mask, guava, np.nan)
ohia_masked = np.where(final_mask, ohia, np.nan)

# Flatten data for classification
guava_flat = guava_masked.flatten()
ohia_flat = ohia_masked.flatten()

print(f"Raster shape: {guava.shape}")
print(f"Valid pixels after masking: {final_mask.sum():,}")

Raster shape: (1842, 2151)
Valid pixels after masking: 1,184,676


# Load Sentinel Data

In [5]:
sent_dat = []

for path in sentinel_path:
    with rio.open(path) as src:
        sentinel = src.read()
        profile = src.profile
    # Apply spatial mask
    sentinel_masked = np.where(
        final_mask[None, :, :],
        sentinel,
        np.nan
    )

    sent_dat.append(sentinel_masked)
    

sent_19, sent_25 = sent_dat

# flatten sentinel data
n_bands, rows, cols = sent_19.shape
sent_19_flat = sent_19.reshape(n_bands, -1).T
sent_25_flat = sent_25.reshape(n_bands, -1).T


print(f"Number of bands: {n_bands}")
print(f"2019 data shape: {sent_19_flat.shape}")
print(f"2025 data shape: {sent_25_flat.shape}")

Number of bands: 13
2019 data shape: (3962142, 13)
2025 data shape: (3962142, 13)


In [7]:
# Build cleaned dataset

# column names
band_cols = [f"spec_{i}" for i in range(n_bands)]

# build sentinel dataframes
df = pd.DataFrame(sent_19_flat, columns=band_cols)
df["guava"] = guava_flat
df["ohia"] = ohia_flat

df25 = pd.DataFrame(sent_25_flat, columns=band_cols)
df25["guava"] = guava_flat
df25["ohia"] = ohia_flat

# add coordinates
row_indices, col_indices = np.indices((rows, cols))

df["row"] = row_indices.flatten()
df["col"] = col_indices.flatten()

df25["row"] = row_indices.flatten()
df25["col"] = col_indices.flatten()

# clean
df_clean = df.dropna()
df_clean_25 = df25.dropna()


# Sentinel Classification Models

GAO species classifications are the "ground truth" for training. Tests across multiple thresholds.

In [8]:
# Guava cover thresholds for training data
guava_thresholds = [0.6, 0.7, 0.8, 0.9]
lower_thresholds = [0.2, 0.3, 0.4]

# Probability threshold
desired_precisions = [0.85]
max_fpr = 0.05

# Other parameters
test_size = 0.3
random_state = 42
ground_truth_threshold = 0.6
balance_classes = True
max_per_class = 20000



In [10]:
# Create benchmark to compare models against

benchmark_df = df_clean.copy()

benchmark_df['benchmark_label'] = np.where(
    benchmark_df['guava'] > ground_truth_threshold,
    1,
    0
)

n_benchmark_per_class = 5000

benchmark_df = pd.concat([
    benchmark_df[
        benchmark_df['benchmark_label'] == 0
    ].sample(
        n=n_benchmark_per_class,
        random_state=123
    ),

    benchmark_df[
        benchmark_df['benchmark_label'] == 1
    ].sample(
        n=n_benchmark_per_class,
        random_state=123
    )
], ignore_index=True)


benchmark_coords = set(
    zip(
        benchmark_df['row'],
        benchmark_df['col']
    )
)

drop_cols_benchmark = [
    'guava',
    'benchmark_label',
    'row',
    'col',
    'ohia'
]

X_benchmark = benchmark_df.drop(
    columns=drop_cols_benchmark
).values

y_benchmark = benchmark_df[
    'benchmark_label'
].values


print(f"Benchmark features: {X_benchmark.shape}")
print(f"Benchmark labels: {y_benchmark.shape}")

Benchmark features: (10000, 13)
Benchmark labels: (10000,)


In [11]:
# Preprocessing data

def brightness_normalize(X):
    brightness = X.sum(axis=1, keepdims=True)
    brightness[np.abs(brightness) < 1e-12] = 1e-6
    return X / brightness

def preprocess_features(X, scaler=None, fit_scaler=False):
    spectral = X[:, :10] # First 10 columns are spectral bands
    indices = X[:, 10:] # Remaining columns are indices

    # Brightness-normalize spectral bands only
    spectral_bn = brightness_normalize(spectral)

    # Recombine
    X_processed = np.hstack([
        spectral_bn,
        indices
    ])

    # Standardize everything
    if fit_scaler:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_processed)
        return X_scaled, scaler

    else:
        X_scaled = scaler.transform(X_processed)
        return X_scaled

In [14]:
def create_binary_training_data(upper, lower, df_clean = df_clean, df_clean_25 = df_clean_25,):
      
    binary_df = df_clean[
        (df_clean['guava'] > upper) |
        (df_clean['guava'] <= lower)
    ].copy()

    binary_df = binary_df[
        ~binary_df.set_index(['row', 'col']).index.isin(
            benchmark_df.set_index(['row', 'col']).index
        )
    ].copy()

    binary_df['label'] = np.where(
        binary_df['guava'] > upper,
        1,
        0
    )

    # Add high-confidence guava pixels from 2025
    binary_25 = df_clean_25[
        df_clean_25['guava'] > 0.9
    ].copy()

    binary_25 = binary_25[
        ~binary_25.set_index(['row', 'col']).index.isin(
            benchmark_df.set_index(['row', 'col']).index
        )
    ].copy()


    binary_25['label'] = 1

    binary_df = pd.concat(
        [binary_df, binary_25],
        ignore_index=True
    )

    print("\nOriginal class counts:")
    print(binary_df['label'].value_counts())
    
    return binary_df

In [17]:
def balance_classes(
    binary_df,
    max_per_class=20000,
    random_state=42
):

    class_counts = binary_df['label'].value_counts()

    min_class_size = class_counts.min()

    n_per_class = min(
        max_per_class,
        min_class_size
    )

    balanced_df = pd.concat([

        binary_df[
            binary_df['label'] == 0
        ].sample(
            n=n_per_class,
            random_state=random_state
        ),

        binary_df[
            binary_df['label'] == 1
        ].sample(
            n=n_per_class,
            random_state=random_state
        )

    ], ignore_index=True)

    # Shuffle reproducibly
    balanced_df = balanced_df.sample(
        frac=1,
        random_state=random_state
    ).reset_index(drop=True)

    print("\nBalanced class counts:")
    print(balanced_df['label'].value_counts())
    
    drop_cols = [
        'guava',
        'label',
        'row',
        'col',
        'ohia'
    ]

    X = balanced_df.drop(
        columns=drop_cols
    ).values

    y = balanced_df[
        'label'
    ].values

    return balanced_df, X, y

In [18]:
def split_and_preprocess(X, y):

    # 70% train
    # 15% validation
    # 15% test

    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        stratify=y,
        test_size=test_size,
        random_state=random_state
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        stratify=y_temp,
        test_size=0.5,
        random_state=random_state
    )

    print(
        f"\nN train = {len(y_train)}, "
        f"N validation = {len(y_val)}, "
        f"N test = {len(y_test)}"
    )


    X_train_scaled, scaler = preprocess_features(
        X_train,
        fit_scaler=True
    )

    X_val_scaled = preprocess_features(
        X_val,
        scaler=scaler
    )

    X_test_scaled = preprocess_features(
        X_test,
        scaler=scaler
    )
    
    return X_train_scaled, scaler, y_train, X_val_scaled, y_val, X_test_scaled,  y_test


In [20]:
def run_svm(X_train_scaled, y_train, random_state):
    
    svc = SVC(
        probability=False,
        random_state=random_state
    )

    param_grid = {
        'C': [0.1, 1, 10],
        'kernel': ['rbf'],
        'gamma': ['scale', 'auto']
    }

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=random_state
    )

    grid_search = GridSearchCV(
        svc,
        param_grid,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1
    )

    grid_search.fit(
        X_train_scaled,
        y_train
    )


    best_params = grid_search.best_params_

    best_svc = SVC(
        C=best_params['C'],
        gamma=best_params['gamma'],
        kernel='rbf',
        probability=True,
        random_state=random_state
    )

    best_svc.fit(X_train_scaled, y_train)

    print("\nBest SVM parameters:")
    print(grid_search.best_params_)
    
    return best_svc

In [21]:
def run_rf(X_train_scaled, y_train, random_state):
    
    rf = RandomForestClassifier(
        random_state=random_state,
        n_jobs=-1
    )

    param_grid = {

        'n_estimators': [
            300,
            500
        ],

        'max_depth': [
            None,
            20
        ],

        'min_samples_leaf': [
            1,
            2
        ],

        'max_features': [
            'sqrt'
        ]
    }


    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=random_state
    )

    grid_search = GridSearchCV(
        rf,
        param_grid,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(
        X_train_scaled,
        y_train
    )

    best_rf = grid_search.best_estimator_

    print("\nBest RF parameters:")
    print(grid_search.best_params_)
    
    return best_rf

In [24]:
def get_threshold(classifier,  X_val_scaled, y_val, X_test_scaled, y_test,max_fpr = 0.05, desired_precision = 0.85):

    y_proba_val = classifier.predict_proba(
        X_val_scaled
    )[:, 1]

    precision_curve, recall_curve, thresholds = (
        precision_recall_curve(
            y_val,
            y_proba_val
        )
    )


    candidate_thresholds = []

    for idx, thresh in enumerate(thresholds):

        y_pred_temp = (
            y_proba_val >= thresh
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_val,
            y_pred_temp,
            labels=[0, 1]
        ).ravel()

        fpr = (
            fp / (fp + tn)
            if (fp + tn) > 0
            else 0
        )

        precision_here = precision_curve[idx]
        recall_here = recall_curve[idx]

        # Require BOTH:
        # precision >= desired precision
        # FPR <= maximum allowable FPR
        
        if (
            precision_here >= desired_precision
            and fpr <= max_fpr
        ):

            candidate_thresholds.append({
                'threshold': thresh,
                'precision': precision_here,
                'recall': recall_here,
                'fpr': fpr
            })


    # Choose qualifying threshold with highest recall
    if len(candidate_thresholds) > 0:

        best_candidate = max(
            candidate_thresholds,
            key=lambda x: x['recall']
        )

        best_threshold = best_candidate['threshold']

    else:

        raise ValueError(
            f"No probability threshold achieved both "
            f"precision >= {desired_precision:.2f} "
            f"and FPR <= {max_fpr:.2f}"
        )


    print(
        f"\nDesired precision: "
        f"{desired_precision:.2f}"
    )

    print(
        f"Maximum FPR: "
        f"{max_fpr:.2f}"
    )

    print(
        f"Selected probability threshold: "
        f"{best_threshold:.4f}"
    )

    print(
        f"Validation precision at threshold: "
        f"{best_candidate['precision']:.3f}"
    )

    print(
        f"Validation recall at threshold: "
        f"{best_candidate['recall']:.3f}"
    )

    print(
        f"Validation FPR at threshold: "
        f"{best_candidate['fpr']:.3f}"
    )


    # Validation Metrics

    y_pred_val = (
        y_proba_val >= best_threshold
    ).astype(int)

    tn_val, fp_val, fn_val, tp_val = confusion_matrix(
        y_val,
        y_pred_val,
        labels=[0, 1]
    ).ravel()

    val_fpr = fp_val / (fp_val + tn_val)

    val_acc = accuracy_score(
        y_val,
        y_pred_val
    )

    val_precision = precision_score(
        y_val,
        y_pred_val,
        zero_division=0
    )

    val_recall = recall_score(
        y_val,
        y_pred_val,
        zero_division=0
    )

    val_f1 = f1_score(
        y_val,
        y_pred_val,
        zero_division=0
    )


    print(
        f"Validation accuracy: "
        f"{val_acc:.3f}"
    )

    print(
        f"Validation precision: "
        f"{val_precision:.3f}"
    )

    print(
        f"Validation recall: "
        f"{val_recall:.3f}"
    )

    print(
        f"Validation F1: "
        f"{val_f1:.3f}"
    )

    print(
        f"Validation FPR: "
        f"{val_fpr:.3f}"
    )


    # Test Set
    
    y_proba_test = classifier.predict_proba(
        X_test_scaled
    )[:, 1]


    y_pred_test = (
        y_proba_test >= best_threshold
    ).astype(int)

    tn_test, fp_test, fn_test, tp_test = confusion_matrix(
        y_test,
        y_pred_test,
        labels=[0, 1]
    ).ravel()

    test_fpr = fp_test / (fp_test + tn_test)


    test_acc = accuracy_score(
        y_test,
        y_pred_test
    )

    test_precision = precision_score(
        y_test,
        y_pred_test,
        zero_division=0
    )

    test_recall = recall_score(
        y_test,
        y_pred_test,
        zero_division=0
    )

    test_f1 = f1_score(
        y_test,
        y_pred_test,
        zero_division=0
    )


    print("\nFINAL TEST SET")

    print(
        f"Accuracy:  {test_acc:.3f}"
    )

    print(
        f"Precision: {test_precision:.3f}"
    )

    print(
        f"Recall:    {test_recall:.3f}"
    )

    print(
        f"F1:        {test_f1:.3f}"
    )

    print(
        f"FPR:       {test_fpr:.3f}"
    )

    print("\nClassification report:")

    print(
        classification_report(
            y_test,
            y_pred_test,
            zero_division=0
        )
    )

    return best_threshold, val_acc, val_precision, val_recall, val_f1,val_fpr,test_acc, test_precision, test_recall, test_f1,test_fpr




In [42]:
def apply_model(balanced_df, guava_masked, sentinel_path, scaler, classifier, model, upper, lower):
    
    used_mask = np.zeros(
        guava_masked.shape,
        dtype=bool
    )


    rows = balanced_df[
        'row'
    ].astype(int).values

    cols = balanced_df[
        'col'
    ].astype(int).values


    valid_coords = (
        (rows >= 0) &
        (rows < used_mask.shape[0]) &
        (cols >= 0) &
        (cols < used_mask.shape[1])
    )


    used_mask[
        rows[valid_coords],
        cols[valid_coords]
    ] = True


    # Apply model 
    
    pred_raster_19 = None
    profile_19 = None

    for s in sentinel_path:

        yr = os.path.basename(
            s
        ).split('_')[1].split('.')[0]


        print(
            f"\nPredicting Sentinel year: {yr}"
        )


        with rio.open(s) as ref:

            profile = ref.profile.copy()

            sent = (
                ref.read()
                .astype(float)
            )


        profile.update(
            dtype=rio.float32,
            count=1,
            compress='lzw',
            nodata=np.nan
        )


        # ==============================================
        # Apply spatial mask
        # ==============================================

        sent_masked = np.where(
            final_mask[None, :, :],
            sent,
            np.nan
        )


        bands, height, width = (
            sent_masked.shape
        )


        reshaped = (
            sent_masked
            .reshape(
                bands,
                -1
            )
            .T
        )


        valid_pixels = np.all(
            np.isfinite(
                reshaped
            ),
            axis=1
        )


        preds = np.full(
            reshaped.shape[0],
            np.nan,
            dtype=float
        )


        # Preprocess image

        X_raster = reshaped[
            valid_pixels
        ]


        X_raster_scaled = preprocess_features(
            X_raster,
            scaler=scaler
        )


        # Predict

        preds_valid = (
            classifier.predict_proba(
                X_raster_scaled
            )[:, 1]
        )


        preds[
            valid_pixels
        ] = preds_valid


        pred_raster_full = (
            preds.reshape(
                height,
                width
            )
        )


        # Save Map

        output_prob_path = (
            f"sent_pred/"
            f"guava_{model}_prob_{yr}_"
            f"U{upper}_L{lower}.tif"
        )


        with rio.open(
            output_prob_path,
            'w',
            **profile
        ) as dst:

            dst.write(
                pred_raster_full.astype(
                    rio.float32
                ),
                1
            )


        print(
            f"{output_prob_path} written"
        )


        if str(yr) in [
            '19',
            '2019'
        ]:

            pred_raster_19 = (
                pred_raster_full.copy()
            )

            profile_19 = (
                profile.copy()
            )
            
    return pred_raster_19, profile_19, used_mask


In [51]:
def error_map(model, upper, lower, guava_masked, pred_raster_19, used_mask, best_threshold, profile_19):

    valid_gt_mask = np.isfinite(
        guava_masked
    )

    gt_bin = np.full(
        guava_masked.shape,
        np.nan,
        dtype=float
    )

    gt_bin[
        valid_gt_mask
    ] = (
        guava_masked[
            valid_gt_mask
        ] > ground_truth_threshold
    ).astype(float)

    # Binary prediction
    
    pred_mask = np.isfinite(
        pred_raster_19
    )

    combined_mask = (
        pred_mask &
        valid_gt_mask &
        ~used_mask
    )

    pred_binary = np.full(
        pred_raster_19.shape,
        np.nan,
        dtype=float
    )


    pred_binary[
        pred_mask
    ] = (
        pred_raster_19[
            pred_mask
        ] >= best_threshold
    ).astype(float)

    # Error map

    err_map = np.full(
        pred_raster_19.shape,
        np.nan,
        dtype=float
    )


    err_map[
        combined_mask
    ] = (
        pred_binary[
            combined_mask
        ] !=
        gt_bin[
            combined_mask
        ]
    ).astype(float)


    error_path = (
        f"sent_pred/"
        f"guava_{model}_error_19_"
        f"U{upper}_L{lower}.tif"
    )


    error_profile = (
        profile_19.copy()
    )


    error_profile.update(
        dtype=rio.float32,
        count=1,
        compress='lzw',
        nodata=np.nan
    )


    with rio.open(
        error_path,
        'w',
        **error_profile
    ) as dst:

        dst.write(
            err_map.astype(
                rio.float32
            ),
            1
        )


    print(
        f"{error_path} written"
    )


In [56]:
def model_guava(model = 'rf', 
                guava_thresholds = guava_thresholds, lower_thresholds = lower_thresholds,
               df_clean = df_clean, df_clean_25 = df_clean_25,
               random_state = random_state,
               guava_masked = guava_masked, 
               sentinel_path = sentinel_path, 
               ground_truth_threshold = ground_truth_threshold):
    
    accuracy_records = []
    for upper in guava_thresholds:
        for lower in lower_thresholds:
            

            print("\n" + "=" * 70)
            print(f"RF: upper={upper}, lower={lower}")
            print("=" * 70)

 
            binary_df = create_binary_training_data( upper = upper, lower = lower)

            balanced_df, X, y = balance_classes(
                binary_df = binary_df,
                max_per_class=20000,
                random_state=42
            )

            X_train_scaled, scaler, y_train, X_val_scaled, y_val, X_test_scaled,  y_test = split_and_preprocess(X, y)

            if model == 'rf':
                classifier = run_rf(X_train_scaled = X_train_scaled, y_train =y_train, random_state = random_state)
                
            elif model == 'svm':
                classifier = run_svm(X_train_scaled = X_train_scaled, y_train =y_train, random_state = random_state)
            

            best_threshold, val_acc, val_precision, val_recall, val_f1,val_fpr,test_acc, test_precision, test_recall, test_f1,test_fpr = get_threshold(classifier = classifier, X_val_scaled = X_val_scaled, y_val = y_val, X_test_scaled = X_test_scaled, y_test = y_test)

            pred_raster_19, profile_19, used_mask = apply_model(balanced_df = balanced_df, guava_masked = guava_masked, sentinel_path = sentinel_path, scaler = scaler, classifier = classifier, model = model, upper = upper, lower = lower)

            # Save Model

            model_filename = (
                f"models/"
                f"guava_{model}_pred_"
                f"U{upper}_L{lower}.pkl"
            )


            with open(
                model_filename,
                'wb'
            ) as f:

                pickle.dump(
                    classifier,
                    f
                )


            scaler_filename = (
                f"models/"
                f"guava_{model}_pred_"
                f"U{upper}_L{lower}_scaler.pkl"
            )


            with open(
                scaler_filename,
                'wb'
            ) as f:

                pickle.dump(
                    scaler,
                    f
                )



            error_map(model = model, upper = upper, lower = lower, guava_masked = guava_masked, pred_raster_19 = pred_raster_19, used_mask = used_mask, best_threshold = best_threshold, profile_19=profile_19)
            
            # Compare against benchmark   
            X_benchmark_scaled = preprocess_features(
                X_benchmark,
                scaler=scaler
            )
            
            benchmark_proba = classifier.predict_proba(
                X_benchmark_scaled
            )[:, 1]

            benchmark_pred = (
                benchmark_proba >= best_threshold
            ).astype(int)

            benchmark_acc = accuracy_score(
                y_benchmark,
                benchmark_pred
            )

            benchmark_precision = precision_score(
                y_benchmark,
                benchmark_pred,
                zero_division=0
            )

            benchmark_recall = recall_score(
                y_benchmark,
                benchmark_pred,
                zero_division=0
            )

            benchmark_f1 = f1_score(
                y_benchmark,
                benchmark_pred,
                zero_division=0
            )
            
            # Save Results

            accuracy_records.append({

                'upper_threshold':
                    upper,

                'lower_threshold':
                    lower,

                'pred_threshold':
                    best_threshold,

                'guava_threshold':
                    ground_truth_threshold,

                'model':
                    model,
                
                'benchmark_acc': benchmark_acc,
                'benchmark_precision': benchmark_precision,
                'benchmark_recall': benchmark_recall,
                'benchmark_f1': benchmark_f1,
                
                'max_fpr_target': max_fpr,

                'val_fpr': val_fpr,

                'test_fpr': test_fpr,



                # Validation
                'val_acc':
                    val_acc,

                'val_precision':
                    val_precision,

                'val_recall':
                    val_recall,

                'val_f1':
                    val_f1,

                # Independent test
                'test_acc':
                    test_acc,

                'test_precision':
                    test_precision,

                'test_recall':
                    test_recall,

                'test_f1':
                    test_f1,


            })


    return accuracy_records



In [57]:
accuracy_rf = pd.DataFrame(model_guava(model = 'rf'))
accuracy_svm = pd.DataFrame(model_guava(model = 'svm'))



# Combine RF and SVM

accuracy_df = pd.concat(
    [
        accuracy_rf,
        accuracy_svm
    ],
    ignore_index=True
)

print(
    "\nCombined RF + SVM summary:"
)

print(
    accuracy_df
)


RF: upper=0.6, lower=0.2

Original class counts:
label
0    732176
1     85551
Name: count, dtype: int64

Balanced class counts:
label
1    20000
0    20000
Name: count, dtype: int64

N train = 28000, N validation = 6000, N test = 6000
Fitting 3 folds for each of 8 candidates, totalling 24 fits

Best RF parameters:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'n_estimators': 500}

Desired precision: 0.85
Maximum FPR: 0.05
Selected probability threshold: 0.7527
Validation precision at threshold: 0.938
Validation recall at threshold: 0.757
Validation FPR at threshold: 0.050
Validation accuracy: 0.854
Validation precision: 0.938
Validation recall: 0.757
Validation F1: 0.838
Validation FPR: 0.050

FINAL TEST SET
Accuracy:  0.846
Precision: 0.931
Recall:    0.747
F1:        0.829
FPR:       0.055

Classification report:
              precision    recall  f1-score   support

           0       0.79      0.94      0.86      3000
           1       0.93      0.75      0.

sent_pred/guava_rf_prob_19_U0.7_L0.4.tif written

Predicting Sentinel year: 25
sent_pred/guava_rf_prob_25_U0.7_L0.4.tif written
sent_pred/guava_rf_error_19_U0.7_L0.4.tif written

RF: upper=0.8, lower=0.2

Original class counts:
label
0    732176
1     45020
Name: count, dtype: int64

Balanced class counts:
label
1    20000
0    20000
Name: count, dtype: int64

N train = 28000, N validation = 6000, N test = 6000
Fitting 3 folds for each of 8 candidates, totalling 24 fits

Best RF parameters:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 300}

Desired precision: 0.85
Maximum FPR: 0.05
Selected probability threshold: 0.7200
Validation precision at threshold: 0.943
Validation recall at threshold: 0.825
Validation FPR at threshold: 0.050
Validation accuracy: 0.887
Validation precision: 0.943
Validation recall: 0.825
Validation F1: 0.880
Validation FPR: 0.050

FINAL TEST SET
Accuracy:  0.889
Precision: 0.941
Recall:    0.829
F1:        0.882
FPR:       0.


Best RF parameters:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'n_estimators': 500}

Desired precision: 0.85
Maximum FPR: 0.05
Selected probability threshold: 0.7529
Validation precision at threshold: 0.939
Validation recall at threshold: 0.775
Validation FPR at threshold: 0.050
Validation accuracy: 0.862
Validation precision: 0.939
Validation recall: 0.775
Validation F1: 0.849
Validation FPR: 0.050

FINAL TEST SET
Accuracy:  0.858
Precision: 0.930
Recall:    0.774
F1:        0.845
FPR:       0.058

Classification report:
              precision    recall  f1-score   support

           0       0.81      0.94      0.87      3000
           1       0.93      0.77      0.84      3000

    accuracy                           0.86      6000
   macro avg       0.87      0.86      0.86      6000
weighted avg       0.87      0.86      0.86      6000


Predicting Sentinel year: 19
sent_pred/guava_rf_prob_19_U0.9_L0.4.tif written

Predicting Sentinel year: 25
sent_pred/g


RF: upper=0.7, lower=0.4

Original class counts:
label
0    808619
1     64090
Name: count, dtype: int64

Balanced class counts:
label
1    20000
0    20000
Name: count, dtype: int64

N train = 28000, N validation = 6000, N test = 6000

Best SVM parameters:
{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}

Desired precision: 0.85
Maximum FPR: 0.05
Selected probability threshold: 0.8686
Validation precision at threshold: 0.931
Validation recall at threshold: 0.671
Validation FPR at threshold: 0.050
Validation accuracy: 0.810
Validation precision: 0.931
Validation recall: 0.671
Validation F1: 0.780
Validation FPR: 0.050

FINAL TEST SET
Accuracy:  0.806
Precision: 0.927
Recall:    0.664
F1:        0.774
FPR:       0.052

Classification report:
              precision    recall  f1-score   support

           0       0.74      0.95      0.83      3000
           1       0.93      0.66      0.77      3000

    accuracy                           0.81      6000
   macro avg       0.83      0.81 

sent_pred/guava_svm_prob_19_U0.9_L0.3.tif written

Predicting Sentinel year: 25
sent_pred/guava_svm_prob_25_U0.9_L0.3.tif written
sent_pred/guava_svm_error_19_U0.9_L0.3.tif written

RF: upper=0.9, lower=0.4

Original class counts:
label
0    808619
1     28091
Name: count, dtype: int64

Balanced class counts:
label
1    20000
0    20000
Name: count, dtype: int64

N train = 28000, N validation = 6000, N test = 6000

Best SVM parameters:
{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}

Desired precision: 0.85
Maximum FPR: 0.05
Selected probability threshold: 0.8322
Validation precision at threshold: 0.941
Validation recall at threshold: 0.805
Validation FPR at threshold: 0.050
Validation accuracy: 0.877
Validation precision: 0.941
Validation recall: 0.805
Validation F1: 0.868
Validation FPR: 0.050

FINAL TEST SET
Accuracy:  0.876
Precision: 0.934
Recall:    0.808
F1:        0.867
FPR:       0.057

Classification report:
              precision    recall  f1-score   support

           0    

In [59]:
top5 = (
    accuracy_df
    .sort_values(
        'benchmark_f1',
        ascending=False
    )
    .head(5)
)

top5

In [62]:
# Save model results
accuracy_df.to_csv('models/accuracy_df.csv')
# accuracy_df = pd.read_csv('models/accuracy_df.csv')



# Ensemble Predictions 

In [87]:
binary_predictions_2019 = []
binary_predictions_2025 = []

for _, row in top5.iterrows():

    upper = row['upper_threshold']
    lower = row['lower_threshold']
    model = row['model']
    thresh = row['pred_threshold']

    print(f"Loading {model} | U{upper} L{lower}")

    # -----------------------------
    # Load rasters
    # -----------------------------
    with rio.open(
        f"sent_pred/guava_{model}_prob_19_U{upper}_L{lower}.tif"
    ) as src:
        guava19 = src.read(1)
        profile19 = src.profile.copy()

    with rio.open(
        f"sent_pred/guava_{model}_prob_25_U{upper}_L{lower}.tif"
    ) as src:
        guava25 = src.read(1)
        profile25 = src.profile.copy()

    # -----------------------------
    # Convert probabilities to binary
    # using model-specific threshold
    # -----------------------------
    pred19 = np.where(
        np.isnan(guava19),
        np.nan,
        (guava19 > thresh).astype(float)
    )

    pred25 = np.where(
        np.isnan(guava25),
        np.nan,
        (guava25 > thresh).astype(float)
    )
    
    # -----------------------------
    # Remove problematic region
    # -----------------------------
    pred19[mask_bad_data] = np.nan
    pred25[mask_bad_data] = np.nan

    binary_predictions_2019.append(pred19)
    binary_predictions_2025.append(pred25)


# -----------------------------
# Stack models
# -----------------------------
stack19 = np.stack(binary_predictions_2019)
stack25 = np.stack(binary_predictions_2025)


# -----------------------------
# Ensemble proportion
# proportion of valid models
# classifying each pixel as guava
# -----------------------------
valid_counts19 = np.sum(~np.isnan(stack19), axis=0)
valid_counts25 = np.sum(~np.isnan(stack25), axis=0)

ensemble_2019 = np.divide(
    np.nansum(stack19, axis=0),
    valid_counts19,
    out=np.full(valid_counts19.shape, np.nan, dtype=float),
    where=valid_counts19 > 0
)

ensemble_2025 = np.divide(
    np.nansum(stack25, axis=0),
    valid_counts25,
    out=np.full(valid_counts25.shape, np.nan, dtype=float),
    where=valid_counts25 > 0
)


# -----------------------------
# Majority-vote binary ensemble
# >50% of valid models must predict guava
# -----------------------------
ensemble_2019_bin = np.where(
    np.isnan(ensemble_2019),
    np.nan,
    (ensemble_2019 > 0.5).astype(float)
)

ensemble_2025_bin = np.where(
    np.isnan(ensemble_2025),
    np.nan,
    (ensemble_2025 > 0.5).astype(float)
)



# -----------------------------
# Save ensemble maps
# -----------------------------
profile19.update(
    dtype='float32',
    count=1,
    nodata=np.nan
)

profile25.update(
    dtype='float32',
    count=1,
    nodata=np.nan
)

with rio.open(
    "sent_pred/guava_ensemble_binary_2019.tif",
    "w",
    **profile19
) as dst:
    dst.write(ensemble_2019_bin.astype("float32"), 1)

with rio.open(
    "sent_pred/guava_ensemble_binary_2025.tif",
    "w",
    **profile25
) as dst:
    dst.write(ensemble_2025_bin.astype("float32"), 1)

Loading svm | U0.7 L0.2
Loading svm | U0.6 L0.2
Loading svm | U0.8 L0.2
Loading rf | U0.7 L0.2
Loading rf | U0.6 L0.2
